# Gold Layer
**Entrada:** tabelas `silver.tb_*` · **Saída:** tabelas `gold.dim_*`, `gold.bridge_*` e `gold.fact_movies_performance`

| Tabela | Papel | Grão |
|---|---|---|
| `fact_movies_performance` | Fato: métricas financeiras e de engajamento | 1 por filme **lançado** |
| `dim_movies` | Dimensão central do filme | 1 por filme |
| `dim_genres` | Catálogo de gêneros | 1 por gênero |
| `dim_people` | Atores, diretores e roteiristas | 1 por (nome, tipo) |
| `dim_companies` | Produtoras | 1 por produtora |
| `dim_reviews` | Resumo das avaliações dos usuários | 1 por filme com avaliações |
| `bridge_movie_*` | Relações N:N entre filme e gênero, pessoa ou produtora | 1 por par de chaves |

## Decisões de projeto
- **Chaves substitutas com `row_number()`** ordenado pela chave natural. Diferente de `monotonically_increasing_id()`, o resultado é **determinístico**: recriar a tabela gera os mesmos `sk_*`, o que mantém as FKs das pontes e da fato consistentes.
- **Cada tabela é gravada e relida** antes de ser usada na seguinte. Assim todas as FKs apontam para as chaves que estão realmente gravadas, e a janela de numeração não é recalculada a cada uso.
- **Tabelas-ponte** evitam duplicar linhas da fato: sem elas, juntar a fato com gêneros ou atores multiplicaria a receita.
- **Fato só de filmes lançados** (`status_filme = 'Lançado'`), como pede o enunciado. A `dim_movies` mantém todos os filmes.
- **Tipos exatos do enunciado** (`DECIMAL(18,2)`, `BIGINT`, `INT`, `DOUBLE`), conferidos na validação final.

# 0. Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "atividade_2_cinedata"
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")


def gravar_gold(df, tabela):
    # Delta + overwrite: o notebook pode ser reexecutado sem duplicar dados.
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabela))
    return spark.table(tabela)


def com_sk(df, nome_sk, *ordem):
    return df.withColumn(nome_sk, F.row_number().over(Window.orderBy(*ordem)).cast("bigint"))

## 1. Dimensões
### 1.1 `gold.dim_movies`
Metadados descritivos de **todos** os filmes. A chave natural (`id_filme`) é mantida ao lado da substituta (`sk_movie_id`) para rastrear a origem.

In [0]:
dim_movies = gravar_gold(
    com_sk(spark.table("silver.tb_info_filmes")
           .select("id_filme", "titulo", "data_lancamento", "ano_lancamento",
                   "duracao_minutos", "idioma_original", "status_filme", "sinopse"),
           "sk_movie_id", "id_filme")
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse"),
    "gold.dim_movies")

# Mapa chave natural -> chave substituta, usado nas pontes, na fato e em dim_reviews
mapa_filmes = dim_movies.select("id_filme", "sk_movie_id")

### 1.2 `gold.dim_genres`
Catálogo único: a Silver já traz um registro por (filme, gênero), então basta o `distinct` do nome.

In [0]:
dim_genres = gravar_gold(
    com_sk(spark.table("silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct(),
           "sk_genre_id", "nome_genero")
    .select("sk_genre_id", "nome_genero"),
    "gold.dim_genres")

### 1.3 `gold.dim_people` e `gold.dim_companies`
A `silver.tb_pessoas_empresas` é unificada; aqui ela se divide pelo `tipo_entidade`:
- **Pessoas físicas** (`Ator`, `Diretor`, `Roteirista`) → `dim_people`, com o tipo em `tipo_pessoa`. Quem atua **e** dirige aparece em duas linhas, uma por tipo, como o enunciado define.
- **Produtoras** → `dim_companies`.


In [0]:
silver_entidades = spark.table("silver.tb_pessoas_empresas")
TIPOS_PESSOA = ["Ator", "Diretor", "Roteirista"]

dim_people = gravar_gold(
    com_sk(silver_entidades.where(F.col("tipo_entidade").isin(TIPOS_PESSOA))
           .select(F.col("nome_entidade").alias("nome_pessoa"),
                   F.col("tipo_entidade").alias("tipo_pessoa")).distinct(),
           "sk_person_id", "tipo_pessoa", "nome_pessoa")
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa"),
    "gold.dim_people")

dim_companies = gravar_gold(
    com_sk(silver_entidades.where(F.col("tipo_entidade") == "Produtora")
           .select(F.col("nome_entidade").alias("nome_produtora")).distinct(),
           "sk_company_id", "nome_produtora")
    .select("sk_company_id", "nome_produtora"),
    "gold.dim_companies")

### 1.4 `gold.dim_reviews`
Transforma as avaliações individuais em **uma linha por filme**: quantidade de avaliações e nota média (2 casas decimais).
- `qtd_avaliacoes_usuarios` conta todas as avaliações; `nota_media_usuarios` ignora notas `NULL` (a média só considera notas válidas).
- Avaliações de um `id_filme` que não existe na `dim_movies` são descartadas pelo `inner join` (a validação final informa quantas).

In [0]:
resumo_avaliacoes = (spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")))

dim_reviews = gravar_gold(
    com_sk(resumo_avaliacoes.join(mapa_filmes, "id_filme", "inner"), "sk_review_id", "sk_movie_id")
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"),
    "gold.dim_reviews")

## 2. Tabelas-ponte
Cada ponte traduz as chaves naturais da Silver para as chaves substitutas e remove pares repetidos (`distinct`). Pares cujo filme ou entidade não existe nas dimensões saem pelo `inner join`.

In [0]:
bridge_movie_genre = gravar_gold(
    spark.table("silver.tb_generos")
    .join(mapa_filmes, "id_filme", "inner")
    .join(dim_genres, F.col("genero") == F.col("nome_genero"), "inner")
    .select("sk_movie_id", "sk_genre_id").distinct(),
    "gold.bridge_movie_genre")

bridge_movie_person = gravar_gold(
    silver_entidades.where(F.col("tipo_entidade").isin(TIPOS_PESSOA))
    .join(mapa_filmes, "id_filme", "inner")
    # a dimensão é única por (nome, tipo): a junção usa as duas colunas
    .join(dim_people, (F.col("nome_entidade") == F.col("nome_pessoa"))
                      & (F.col("tipo_entidade") == F.col("tipo_pessoa")), "inner")
    .select("sk_movie_id", "sk_person_id").distinct(),
    "gold.bridge_movie_person")

bridge_movie_company = gravar_gold(
    silver_entidades.where(F.col("tipo_entidade") == "Produtora")
    .join(mapa_filmes, "id_filme", "inner")
    .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"), "inner")
    .select("sk_movie_id", "sk_company_id").distinct(),
    "gold.bridge_movie_company")

## 3. Fato: `gold.fact_movies_performance`
**Grão:** um registro por filme **lançado**.

Parte da `dim_movies` (filtrada por `status_filme = 'Lançado'`) e faz `left join` com o financeiro e com as métricas. As duas tabelas Silver têm **uma linha por `id_filme`**, então os joins não multiplicam linhas: o grão é preservado (conferido na validação). Filmes sem dados financeiros ou sem métricas continuam na fato, com `NULL` nessas colunas.

In [0]:
financeiro = spark.table("silver.tb_financeiro_filmes")
metricas = spark.table("silver.tb_metricas_engajamento")

fact_movies_performance = gravar_gold(
    dim_movies.where(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")
    .join(financeiro, "id_filme", "left")
    .join(metricas, "id_filme", "left")
    .select("sk_movie_id",
            F.col("orcamento_usd").cast("decimal(18,2)"),
            F.col("receita_usd").cast("decimal(18,2)"),
            F.col("lucro_usd").cast("decimal(18,2)"),
            F.col("orcamento_brl").cast("decimal(18,2)"),
            F.col("receita_brl").cast("decimal(18,2)"),
            F.col("lucro_brl").cast("decimal(18,2)"),
            F.col("popularidade").cast("double"),
            F.col("nota_media_tmdb").cast("double"),
            F.col("qtd_votos_tmdb").cast("int"),
            F.col("nota_media_imdb").cast("double"),
            F.col("qtd_votos_imdb").cast("int")),
    "gold.fact_movies_performance")

## 4. Validação do modelo
**FAIL** (interrompe): esquema idêntico ao enunciado, chaves substitutas únicas, grão da fato preservado, sem pares repetidos nas pontes e **integridade referencial** (nenhuma FK aponta para uma chave inexistente). **QUIET**: volumes por tabela.

In [0]:
# FAIL 1: esquema idêntico ao enunciado (nomes, ordem e tipos)
ESQUEMA = {
  "gold.fact_movies_performance": [("sk_movie_id","bigint"),("orcamento_usd","decimal(18,2)"),("receita_usd","decimal(18,2)"),
      ("lucro_usd","decimal(18,2)"),("orcamento_brl","decimal(18,2)"),("receita_brl","decimal(18,2)"),("lucro_brl","decimal(18,2)"),
      ("popularidade","double"),("nota_media_tmdb","double"),("qtd_votos_tmdb","int"),("nota_media_imdb","double"),("qtd_votos_imdb","int")],
  "gold.dim_movies": [("sk_movie_id","bigint"),("id_filme","string"),("titulo","string"),("data_lancamento","date"),
      ("ano_lancamento","int"),("duracao_minutos","int"),("idioma_original","string"),("status_filme","string"),("sinopse","string")],
  "gold.dim_genres": [("sk_genre_id","bigint"),("nome_genero","string")],
  "gold.dim_people": [("sk_person_id","bigint"),("nome_pessoa","string"),("tipo_pessoa","string")],
  "gold.dim_companies": [("sk_company_id","bigint"),("nome_produtora","string")],
  "gold.dim_reviews": [("sk_review_id","bigint"),("sk_movie_id","bigint"),("qtd_avaliacoes_usuarios","int"),("nota_media_usuarios","double")],
  "gold.bridge_movie_genre": [("sk_movie_id","bigint"),("sk_genre_id","bigint")],
  "gold.bridge_movie_person": [("sk_movie_id","bigint"),("sk_person_id","bigint")],
  "gold.bridge_movie_company": [("sk_movie_id","bigint"),("sk_company_id","bigint")],
}
for tabela, esperado in ESQUEMA.items():
    assert spark.table(tabela).dtypes == esperado, f"{tabela}: esquema diferente do enunciado -> {spark.table(tabela).dtypes}"

# FAIL 2: chaves substitutas únicas e não nulas; grão da fato = 1 linha por filme lançado
def assert_pk(tabela, pk):
    t = spark.table(tabela)
    assert t.where(F.col(pk).isNull()).count() == 0, f"{tabela}: {pk} nula"
    assert t.count() == t.select(pk).distinct().count(), f"{tabela}: {pk} duplicada"

for tabela, pk in [("gold.dim_movies","sk_movie_id"), ("gold.dim_genres","sk_genre_id"), ("gold.dim_people","sk_person_id"),
                   ("gold.dim_companies","sk_company_id"), ("gold.dim_reviews","sk_review_id"),
                   ("gold.fact_movies_performance","sk_movie_id")]:
    assert_pk(tabela, pk)

lancados = spark.table("gold.dim_movies").where("status_filme = 'Lançado'").count()
assert spark.table("gold.fact_movies_performance").count() == lancados, "Grão da fato alterado pelos joins"

# FAIL 3: pontes sem pares repetidos e integridade referencial (nenhuma FK órfã)
def assert_fk(tabela, coluna, pai, pk):
    filha, dim = spark.table(tabela), spark.table(pai)   # um objeto por tabela: a junção referencia colunas deles
    orfas = filha.join(dim, filha[coluna] == dim[pk], "left_anti").count()
    assert orfas == 0, f"{tabela}.{coluna}: {orfas} FK(s) sem correspondente em {pai}"

for ponte, outra, dim_outra in [("gold.bridge_movie_genre","sk_genre_id","gold.dim_genres"),
                                ("gold.bridge_movie_person","sk_person_id","gold.dim_people"),
                                ("gold.bridge_movie_company","sk_company_id","gold.dim_companies")]:
    p = spark.table(ponte)
    assert p.count() == p.dropDuplicates().count(), f"{ponte}: pares repetidos"
    assert_fk(ponte, "sk_movie_id", "gold.dim_movies", "sk_movie_id")
    assert_fk(ponte, outra, dim_outra, outra)

assert_fk("gold.fact_movies_performance", "sk_movie_id", "gold.dim_movies", "sk_movie_id")
assert_fk("gold.dim_reviews", "sk_movie_id", "gold.dim_movies", "sk_movie_id")

# QUIET: volumes
display(spark.createDataFrame([(t, spark.table(t).count()) for t in ESQUEMA], "tabela string, linhas long"))
av_orfas = (spark.table("silver.tb_avaliacoes_usuarios").select("id_filme").distinct()
            .join(spark.table("gold.dim_movies"), "id_filme", "left_anti").count())
print("Filmes com avaliações que não existem em dim_movies (descartados de dim_reviews):", av_orfas)

---
# 5 · Desafio de Analytics
As respostas usam **apenas** as tabelas Gold.

**Janela de tempo (perguntas 5 e 6):** a data limite superior é a **data de lançamento mais recente entre filmes já lançados**, ignorando datas futuras (`<= current_date()`) e filmes não lançados. Os "últimos N anos" vão de `data_limite - N anos` até `data_limite`, ambos inclusive.

In [0]:
data_limite = spark.sql("""
    SELECT MAX(data_lancamento) FROM gold.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
""").first()[0]
print("Data limite superior da base:", data_limite)

### 1. Receita total (em R$) de todos os filmes da base

In [0]:
display(spark.sql("""
    SELECT SUM(receita_brl) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

### 2. Os 5 filmes com maior popularidade

In [0]:
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

### 3. Quantidade de filmes por gênero
`COUNT(DISTINCT sk_movie_id)` conta cada filme uma vez por gênero. Um filme com 3 gêneros aparece nos 3, portanto a soma da coluna é maior que o total de filmes.

In [0]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

### 4. Os 10 filmes de maior receita, com ranking
`RANK()` dá a mesma posição a filmes empatados; por isso o filtro `ranking <= 10` pode devolver mais de 10 linhas se houver empate na décima posição.

In [0]:
display(spark.sql("""
    SELECT ranking, titulo, receita_usd, receita_brl
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS ranking
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    WHERE ranking <= 10
    ORDER BY ranking
"""))

### 5. Ator com mais participações em filmes lançados nos últimos 2 anos
A primeira linha é a resposta; as seguintes permitem ver empates.

In [0]:
display(spark.sql(f"""
    SELECT p.nome_pessoa AS ator, COUNT(DISTINCT b.sk_movie_id) AS participacoes
    FROM gold.bridge_movie_person b
    JOIN gold.dim_people p ON p.sk_person_id = b.sk_person_id
    JOIN gold.dim_movies m ON m.sk_movie_id = b.sk_movie_id
    WHERE p.tipo_pessoa = 'Ator'
      AND m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE'{data_limite}', -24) AND DATE'{data_limite}'
    GROUP BY p.nome_pessoa
    ORDER BY participacoes DESC, ator
    LIMIT 5
"""))

### 6. Produtora com maior lucro nos últimos 5 anos
Soma o lucro dos filmes de cada produtora. Filmes sem orçamento ou receita têm lucro `NULL` e não entram na soma. Em coproduções, o lucro do filme é atribuído a cada produtora envolvida.

In [0]:
display(spark.sql(f"""
    SELECT c.nome_produtora,
           SUM(f.lucro_usd) AS lucro_usd,
           SUM(f.lucro_brl) AS lucro_brl,
           COUNT(DISTINCT f.sk_movie_id) AS filmes_com_lucro
    FROM gold.bridge_movie_company b
    JOIN gold.dim_companies c ON c.sk_company_id = b.sk_company_id
    JOIN gold.fact_movies_performance f ON f.sk_movie_id = b.sk_movie_id
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.lucro_usd IS NOT NULL
      AND m.data_lancamento BETWEEN add_months(DATE'{data_limite}', -60) AND DATE'{data_limite}'
    GROUP BY c.nome_produtora
    ORDER BY lucro_usd DESC
    LIMIT 5
"""))

In [0]:
%sql
SELECT COUNT(*) AS filmes_com_popularidade_igual_ao_ano
FROM gold.fact_movies_performance f
JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
WHERE f.popularidade = m.ano_lancamento;

In [0]:
%sql
SELECT m.titulo, m.status_filme, m.data_lancamento, fin.receita_usd
FROM gold.dim_movies m
LEFT JOIN silver.tb_financeiro_filmes fin ON fin.id_filme = m.id_filme
WHERE lower(m.titulo) IN ('avatar', 'titanic', 'star wars: the force awakens');